# Lab # 03 - Task 1 & Task 2

Image Data Preprocessing and Feature Scaling

Dataset: Food image dataset (20 classes: bacon, banana, bread, broccoli,

butter, carrots, cheese, chicken, cucumber, eggs, fish, lettuce, milk,

onions, peppers, potatoes, sausages, spinach, tomato, yogurt)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler

In [ ]:
BASE_DIR = os.getcwd()  # notebooks have no __file__, so use the notebook's folder
DATASET_DIR = os.path.join(BASE_DIR, "train")

In [ ]:
IMG_SIZE = (64, 64)
IMAGES_PER_CLASS = 30

In [ ]:
if not os.path.isdir(DATASET_DIR):
    raise FileNotFoundError(
        f"Could not find the dataset folder at: {DATASET_DIR}\n"
        f"Make sure your extracted 'train' folder sits next to this script, "
        f"or update DATASET_DIR to point to the correct path."
    )

In [ ]:
classes = sorted(os.listdir(DATASET_DIR))
print("Classes found in dataset:", classes)
print("Number of classes:", len(classes))

In [ ]:
images = []
labels = []
corrupted_files = []

In [ ]:
for cls in classes:
    cls_folder = os.path.join(DATASET_DIR, cls)
    file_list = sorted(os.listdir(cls_folder))[:IMAGES_PER_CLASS]

    for fname in file_list:
        fpath = os.path.join(cls_folder, fname)
        try:
            img = Image.open(fpath)
            img.verify()                             
            img = Image.open(fpath).convert("RGB")   
        except (UnidentifiedImageError, OSError, FileNotFoundError):
            corrupted_files.append(fpath)
            continue

        images.append(img)
        labels.append(cls)

In [ ]:
print(f"\nTotal images successfully loaded: {len(images)}")
print(f"Total corrupted/unreadable images skipped: {len(corrupted_files)}")
if corrupted_files:
    print("Corrupted files:", corrupted_files)

In [ ]:
sizes = [img.size for img in images]
print("\nSample original image sizes (width, height):", sizes[:5])

In [ ]:
sample_originals = images[:5]
sample_labels = labels[:5]

In [ ]:
resized_images = [img.resize(IMG_SIZE) for img in images]
sample_resized = resized_images[:5]

In [ ]:
image_array = np.array([np.asarray(img, dtype=np.float32) for img in resized_images])
print("\nShape of preprocessed image array (N, H, W, C):", image_array.shape)

In [ ]:
labels_array = np.array(labels).reshape(-1, 1)
onehot_encoder = OneHotEncoder(sparse_output=False)
labels_onehot = onehot_encoder.fit_transform(labels_array)

In [ ]:
print("\nOne-hot encoded labels shape:", labels_onehot.shape)
print("Encoder categories (order of columns):", onehot_encoder.categories_[0])
print("Example - label:", labels[0], "-> one-hot vector:", labels_onehot[0])

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    axes[0, i].imshow(sample_originals[i])
    axes[0, i].set_title(f"Original\n{sample_labels[i]}\n{sample_originals[i].size}")
    axes[0, i].axis("off")

    axes[1, i].imshow(sample_resized[i])
    axes[1, i].set_title(f"Resized\n{sample_labels[i]}\n{IMG_SIZE}")
    axes[1, i].axis("off")

In [ ]:
plt.suptitle("Original vs Preprocessed (Resized) Images")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "task1_original_vs_resized.png"))
plt.show()

In [ ]:
n_samples = image_array.shape[0]

In [ ]:
flat_features = image_array.reshape(n_samples, -1)
print("\nFlattened feature matrix shape:", flat_features.shape)
print("Raw pixel value range: min =", flat_features.min(), ", max =", flat_features.max())

In [ ]:
standard_scaler = StandardScaler()
features_standardized = standard_scaler.fit_transform(flat_features)

In [ ]:
minmax_scaler = MinMaxScaler()
features_normalized = minmax_scaler.fit_transform(flat_features)

In [ ]:
print("\nAfter StandardScaler -> mean:", round(features_standardized.mean(), 4),
      ", std:", round(features_standardized.std(), 4))
print("After MinMaxScaler   -> min:", features_normalized.min(),
      ", max:", features_normalized.max())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

In [ ]:
axes[0].hist(flat_features.flatten(), bins=50, color="steelblue")
axes[0].set_title("Original Pixel Values\n(range 0-255)")
axes[0].set_xlabel("Pixel value")
axes[0].set_ylabel("Frequency")

In [ ]:
axes[1].hist(features_standardized.flatten(), bins=50, color="salmon")
axes[1].set_title("After StandardScaler\n(mean=0, std=1)")
axes[1].set_xlabel("Standardized value")

In [ ]:
axes[2].hist(features_normalized.flatten(), bins=50, color="seagreen")
axes[2].set_title("After MinMaxScaler\n(range 0-1)")
axes[2].set_xlabel("Normalized value")

In [ ]:
plt.suptitle("Pixel-Value Distributions Before and After Feature Scaling")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "task2_pixel_distributions.png"))
plt.show()

In [ ]:
# MinMax Scaler scales data to fixed value whereas Standard Scaler scales data to variable value
# Feature scaling is bringing value into scalable range